In [1]:
import jax
import numpy as np
import jax.numpy as jnp
from jax import random, jit, grad, jacobian, vmap

In [2]:
# jax arrays are immutable
x = jnp.arange(5)
isinstance(x, jax.Array)

True

In [4]:
y = x.at[1].set(8)
y

Array([0, 8, 2, 3, 4], dtype=int32)

In [3]:
x.devices()

{CudaDevice(id=0)}

In [4]:
def norm(X):
  X = X - X.mean(0)
  return X / X.std(0)

In [6]:
norm_compiled = jit(norm)

In [9]:
np.random.seed(1701)
X = jnp.array(np.random.rand(10000, 10))
np.allclose(norm(X), norm_compiled(X), atol=1E-6)

True

In [10]:
%timeit norm(X).block_until_ready()
%timeit norm_compiled(X).block_until_ready()

279 μs ± 37.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
62.4 μs ± 3.82 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [12]:
def sum_logistic(x):
  return jnp.sum(1.0 / (1.0 + jnp.exp(-x)))

In [ ]:
x_small = jnp.arange(3.)
derivative_fn = grad(sum_logistic)
# print will only show tracer values within jax.jit-decorated code
print(derivative_fn(x_small))

[0.25       0.19661194 0.10499357]


In [14]:
print(grad(jit(grad(jit(grad(sum_logistic)))))(1.0))

-0.0353256


In [20]:
print(jacobian(jnp.exp)(x_small))

[[1.        0.        0.       ]
 [0.        2.7182817 0.       ]
 [0.        0.        7.389056 ]]


In [21]:
key = random.key(1701)
key1, key2 = random.split(key)
mat = random.normal(key1, (150, 100))
batched_x = random.normal(key2, (10, 100))

def apply_matrix(x):
  return jnp.dot(mat, x)

In [24]:
def naively_batched_apply_matrix(v_batched):
  return jnp.stack([apply_matrix(v) for v in v_batched])

print('Naively batched')
%timeit naively_batched_apply_matrix(batched_x).block_until_ready()

Naively batched
The slowest run took 4.62 times longer than the fastest. This could mean that an intermediate result is being cached.
1.65 ms ± 1.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [31]:
@jit
def batched_apply_matrix(batched_x):
  return jnp.dot(batched_x, mat.T)

np.testing.assert_allclose(naively_batched_apply_matrix(batched_x), batched_apply_matrix(batched_x), atol=1E-2, rtol=1E-2)
print('Manually batched')
%timeit batched_apply_matrix(batched_x).block_until_ready()

Manually batched
The slowest run took 34.22 times longer than the fastest. This could mean that an intermediate result is being cached.
895 μs ± 955 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [30]:
@jit
def vmap_batched_apply_matrix(batched_x):
  return vmap(apply_matrix)(batched_x)

np.testing.assert_allclose(naively_batched_apply_matrix(batched_x),
                           vmap_batched_apply_matrix(batched_x), atol=1E-2, rtol=1E-2)
print('Auto-vectorized with vmap')
%timeit vmap_batched_apply_matrix(batched_x).block_until_ready()

Auto-vectorized with vmap
114 μs ± 61.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [5]:
global_list = []

def log2(x):
  global_list.append(x)
  ln_x = jnp.log(x)
  ln_2 = jnp.log(2.0)
  return ln_x / ln_2

print(jax.make_jaxpr(log2)(3.0))

{ lambda ; a:f32[]. let
    b:f32[] = log a
    c:f32[] = log 2.0:f32[]
    d:f32[] = div b c
  in (d,) }


In [8]:
f = lambda x: x**3 + 2*x**2 - 3*x + 1
dfdx = jax.grad(f)
dfdx(2.0)

Array(17., dtype=float32, weak_type=True)

In [9]:
list_of_lists = [
    [1, 2, 3],
    [1, 2],
    [1, 2, 3, 4]
]

jax.tree.map(lambda x: x*2, list_of_lists)

[[2, 4, 6], [2, 4], [2, 4, 6, 8]]

In [11]:
def init_mlp_params(layer_widths):
  params = []
  for n_in, n_out in zip(layer_widths[:-1], layer_widths[1:]):
    params.append(
        dict(weights=np.random.normal(size=(n_in, n_out)) * np.sqrt(2/n_in),
             biases=np.ones(shape=(n_out,))
            )
    )
  return params

params = init_mlp_params([1, 128, 128, 1])

In [14]:
params[0].keys()

dict_keys(['weights', 'biases'])

In [15]:
params[1].keys()

dict_keys(['weights', 'biases'])

In [16]:
jax.tree.map(lambda x: x.shape, params)

[{'biases': (128,), 'weights': (1, 128)},
 {'biases': (128,), 'weights': (128, 128)},
 {'biases': (1,), 'weights': (128, 1)}]